# CURE-OR++ Open-Weight VLM Real-Transfer GPU Matrix

This notebook runs open-weight VLMs over a CURE-OR++ real-transfer prompt pack.
It uses a tiered model matrix so we can smoke-test multiple models before
promoting stable rows to full evaluation.

Default mode: smoke test enabled models from
the configured model matrix.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
import pandas as pd

os.environ.setdefault('HF_HOME', '/tmp/cure_or_pp_hf_home')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Generated by scripts/write_kaggle_vlm_notebook.py.
RUN_MODE = 'full'  # 'smoke' or 'full'
SELECTED_MODEL_SLUGS = ["llava_onevision_qwen2_7b"]  # [] means enabled_by_default models from the matrix.
EMBEDDED_RUNTIME_FILES = {}

PROMPT_PACK_RELATIVE = Path('reports/vlm_api_track_v03_prompt_pack.jsonl')
MODEL_MATRIX_RELATIVE = Path('configs/vlm_open_weight_model_matrix_v03.json')
DATA_RELATIVE = Path('data')
DATASET_CANDIDATES = [
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v03-private'),
    Path('/kaggle/input/datasets/yaroslavkholmirzayev/cure-or-pp-vlm-real-transfer-v03-private'),
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v02-private'),
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v03-private'),
]

def list_input_roots():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return []
    return sorted(path for path in input_root.iterdir() if path.is_dir())

def has_data_payload(path):
    return (
        (path / DATA_RELATIVE).exists()
        or (path / 'data.tar').exists()
        or (path / 'data.zip').exists()
    )

def find_data_source_root():
    input_root = Path('/kaggle/input')
    for path in DATASET_CANDIDATES:
        if has_data_payload(path):
            return path
    if input_root.exists():
        payload_candidates = []
        for pattern in ['data', 'data.tar', 'data.zip']:
            payload_candidates.extend(sorted(input_root.rglob(pattern)))
        print('Data payload candidates:', [str(path) for path in payload_candidates[:20]])
        for candidate in payload_candidates:
            root = candidate.parent if candidate.name != 'data' else candidate.parent
            if has_data_payload(root):
                return root
    return None

def materialize_data_root(source_root):
    if (source_root / DATA_RELATIVE).exists():
        return source_root
    unpack_root = Path('/kaggle/working/cure_or_pp_dataset_unpack')
    if unpack_root.exists():
        shutil.rmtree(unpack_root)
    unpack_root.mkdir(parents=True, exist_ok=True)
    data_tar = source_root / 'data.tar'
    data_zip = source_root / 'data.zip'
    if data_tar.exists():
        with tarfile.open(data_tar) as handle:
            handle.extractall(unpack_root)
        return unpack_root
    if data_zip.exists():
        with zipfile.ZipFile(data_zip) as handle:
            handle.extractall(unpack_root)
        return unpack_root
    raise FileNotFoundError(f'No data payload found under {source_root}')

def unpack_runtime_payloads(source_root, runtime_root):
    for name in ['configs', 'reports', 'scripts']:
        source_dir = source_root / name
        target_dir = runtime_root / name
        if source_dir.exists():
            if target_dir.exists():
                shutil.rmtree(target_dir)
            shutil.copytree(source_dir, target_dir)
            continue
        source_zip = source_root / f'{name}.zip'
        if source_zip.exists():
            with zipfile.ZipFile(source_zip) as handle:
                handle.extractall(runtime_root)

def write_runtime_overrides(source_root, data_root):
    runtime_root = Path('/kaggle/working/cure_or_pp_runtime_pack')
    runtime_root.mkdir(parents=True, exist_ok=True)
    unpack_runtime_payloads(source_root, runtime_root)
    for relative_path, text in EMBEDDED_RUNTIME_FILES.items():
        target = runtime_root / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text, encoding='utf-8')
    runtime_data = runtime_root / DATA_RELATIVE
    if runtime_data.exists() or runtime_data.is_symlink():
        if runtime_data.is_symlink() or runtime_data.is_file():
            runtime_data.unlink()
        else:
            shutil.rmtree(runtime_data)
    runtime_data.symlink_to(data_root / DATA_RELATIVE, target_is_directory=True)
    return runtime_root

print('Python:', sys.version)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not available')
print('Kaggle input roots:', [path.name for path in list_input_roots()])

DATA_SOURCE_ROOT = find_data_source_root()
if DATA_SOURCE_ROOT is None:
    raise FileNotFoundError(
        'Attach the private CURE-OR++ VLM real-transfer dataset to this notebook. '
        f'Checked explicit candidates: {[str(path) for path in DATASET_CANDIDATES]}; '
        f'available /kaggle/input roots: {[path.name for path in list_input_roots()]}'
    )

DATA_ROOT = materialize_data_root(DATA_SOURCE_ROOT)
RUNTIME_ROOT = write_runtime_overrides(DATA_SOURCE_ROOT, DATA_ROOT)
print('DATA_SOURCE_ROOT:', DATA_SOURCE_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RUNTIME_ROOT:', RUNTIME_ROOT)
print('Prompt pack:', RUNTIME_ROOT / PROMPT_PACK_RELATIVE)
print('Model matrix:', RUNTIME_ROOT / MODEL_MATRIX_RELATIVE)


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --index-url https://download.pytorch.org/whl/cu121 "torch==2.5.1+cu121" "torchvision==0.20.1+cu121"
%pip install -q --no-cache-dir -U "transformers>=4.57,<5" "accelerate>=1.8,<2" "pillow<12" num2words "qwen-vl-utils>=0.0.8,<0.1"


In [ ]:
import transformers
import torch

PROMPT_PACK = DATA_ROOT / PROMPT_PACK_RELATIVE
MODEL_MATRIX_PATH = RUNTIME_ROOT / MODEL_MATRIX_RELATIVE
RUNNER = RUNTIME_ROOT / 'scripts/run_hf_vlm.py'
EVALUATOR = RUNTIME_ROOT / 'scripts/evaluate_vlm_response_pack.py'
PROMPT_PACK = RUNTIME_ROOT / PROMPT_PACK_RELATIVE
OUTPUT_ROOT = Path('/kaggle/working/vlm_open_weight_runs') / RUN_MODE
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = Path('/kaggle/temp/cure_or_pp_vlm_cache') / RUN_MODE
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

for required_path in [PROMPT_PACK, MODEL_MATRIX_PATH, RUNNER, EVALUATOR]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

matrix = json.loads(MODEL_MATRIX_PATH.read_text())
if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError(f'RUN_MODE must be smoke or full, got {RUN_MODE!r}')

all_models = matrix['models']
if SELECTED_MODEL_SLUGS:
    selected = [model for model in all_models if model['slug'] in set(SELECTED_MODEL_SLUGS)]
    missing = sorted(set(SELECTED_MODEL_SLUGS) - {model['slug'] for model in selected})
    if missing:
        raise ValueError(f'Unknown SELECTED_MODEL_SLUGS: {missing}')
else:
    selected = [model for model in all_models if model.get('enabled_by_default', False)]

print('transformers:', transformers.__version__)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('torch cuda arch list:', torch.cuda.get_arch_list())
    print('torch device capability:', torch.cuda.get_device_capability())

pd.DataFrame(selected)[['slug', 'model_id', 'tier', 'status']]


In [ ]:
def json_arg(value):
    return json.dumps(value or {}, sort_keys=True)

def run_model(model):
    slug = model['slug']
    limit = int(matrix['smoke_limit'] if RUN_MODE == 'smoke' else matrix['full_limit'])
    run_dir = OUTPUT_ROOT / slug
    run_dir.mkdir(parents=True, exist_ok=True)
    responses = run_dir / 'responses.jsonl'
    model_summary = run_dir / 'model_summary.csv'
    recipe_table = run_dir / 'recipe_table.csv'
    label_table = run_dir / 'label_table.csv'
    audit_table = run_dir / 'audit.csv'

    runner_cmd = [
        sys.executable,
        str(RUNNER),
        '--prompt-pack', str(PROMPT_PACK),
        '--provider', model.get('provider', 'huggingface'),
        '--model', model['model_id'],
        '--model-class', model.get('model_class', 'AutoModelForImageTextToText'),
        '--generation-backend', model.get('generation_backend', 'chat_template'),
        '--image-content-key', model.get('image_content_key', 'path'),
        '--output', str(responses),
        '--cache-dir', str(CACHE_ROOT / slug),
        '--force',
        '--max-tokens', str(model.get('max_tokens', 8)),
        '--dtype', model.get('dtype', 'float16'),
        '--processor-kwargs-json', json_arg(model.get('processor_kwargs')),
        '--model-kwargs-json', json_arg(model.get('model_kwargs')),
        '--generate-kwargs-json', json_arg(model.get('generate_kwargs')),
    ]
    if model.get('revision'):
        runner_cmd.extend(['--revision', model['revision']])
    if model.get('trust_remote_code'):
        runner_cmd.append('--trust-remote-code')
    if model.get('device_map'):
        runner_cmd.extend(['--device-map', model['device_map']])
    if model.get('image_max_side'):
        runner_cmd.extend(['--image-max-side', str(int(model['image_max_side']))])
    if model.get('system_prompt') is not None:
        runner_cmd.extend(['--system-prompt', model['system_prompt']])
    smoke_sample_ids = matrix.get('smoke_sample_ids') if RUN_MODE == 'smoke' else None
    if smoke_sample_ids:
        for sample_id in smoke_sample_ids:
            runner_cmd.extend(['--sample-id', sample_id])
    else:
        runner_cmd.extend(['--limit', str(limit)])

    print('\n=== RUN', slug, model['model_id'], 'limit=', limit, '===')
    print(' '.join(runner_cmd))
    subprocess.run(runner_cmd, check=True, cwd=str(DATA_ROOT))

    eval_cmd = [
        sys.executable,
        str(EVALUATOR),
        '--prompt-pack', str(PROMPT_PACK),
        '--responses', str(responses),
        '--model-summary', str(model_summary),
        '--recipe-table', str(recipe_table),
        '--label-table', str(label_table),
        '--audit-table', str(audit_table),
    ]
    print(' '.join(eval_cmd))
    subprocess.run(eval_cmd, check=True, cwd=str(DATA_ROOT))

    summary = pd.read_csv(model_summary)
    recipe = pd.read_csv(recipe_table)
    label = pd.read_csv(label_table)
    summary.insert(0, 'slug', slug)
    recipe.insert(0, 'slug', slug)
    label.insert(0, 'slug', slug)
    return {'summary': summary, 'recipe': recipe, 'label': label, 'run_dir': run_dir}

results = []
for model in selected:
    try:
        results.append(run_model(model))
    except Exception as exc:
        print(f'FAILED {model["slug"]}: {exc}')
        if RUN_MODE == 'full':
            raise

if not results:
    raise RuntimeError('No successful model runs.')


In [ ]:
summary_df = pd.concat([item['summary'] for item in results], ignore_index=True)
recipe_df = pd.concat([item['recipe'] for item in results], ignore_index=True)
label_df = pd.concat([item['label'] for item in results], ignore_index=True)

summary_out = OUTPUT_ROOT / 'combined_model_summary.csv'
recipe_out = OUTPUT_ROOT / 'combined_recipe_table.csv'
label_out = OUTPUT_ROOT / 'combined_label_table.csv'
summary_df.to_csv(summary_out, index=False)
recipe_df.to_csv(recipe_out, index=False)
label_df.to_csv(label_out, index=False)

manifest = pd.DataFrame([
    {'slug': item['summary']['slug'].iloc[0], 'run_dir': str(item['run_dir'])}
    for item in results
])
manifest_out = OUTPUT_ROOT / 'run_manifest.csv'
manifest.to_csv(manifest_out, index=False)

print('Combined summary:', summary_out)
print('Combined recipe:', recipe_out)
print('Combined labels:', label_out)
print('Manifest:', manifest_out)
summary_df.sort_values(['real_accuracy', 'clean_accuracy'], ascending=False)


In [ ]:
recipe_df.sort_values(['slug', 'recipe'])


In [ ]:
label_df.sort_values(['slug', 'real_accuracy', 'label']).groupby('slug').head(10)
